In [60]:
import requests
import io
from zipfile import ZipFile
import frontmatter


In [ ]:
## patter: https://github.com/{owner}/{repo}/archive/refs/heads/main.zip
## url: https://github.com/evidentlyai/docs/archive/refs/heads/main.zip

In [3]:
repo_owner = "evidentlyai"
repo_name = "docs"
branch_name ="main"
zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'

In [5]:
zip_response = requests.get(zip_url)

In [11]:
zip_archive = ZipFile(io.BytesIO(zip_response.content))

In [12]:
filenames = zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [13]:
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf-8')
print(mdx_content[:150])

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **


In [14]:
post = frontmatter.loads(mdx_content)
print(post.content[:100])

ModuleNotFoundError: No module named 'frontmatter'

In [17]:
!uv add python-frontmatter

Resolved 113 packages in 221ms
Prepared 1 package in 71ms
Installed 1 package in 28ms
 + python-frontmatter==1.3.0


In [18]:


post = frontmatter.loads(mdx_content)
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [19]:
post.metadata

{'title': 'Alerts', 'description': 'How to set up alerts.'}

In [20]:
filename_corrected = filename.split('/', 1)[-1]
print(filename_corrected)

docs/platform/alerts.mdx


In [21]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [22]:
doc

{'content': '<Check>\n  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n![](/images/alerts.png)\n\nTo enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:\n\n* A notification channel.\n\n* An alert condition.\n\n## Notification channels\n\nYou can choose between the following options:\n\n* **Email**. Add email addresses to send alerts to.\n\n* **Slack**. Add a Slack webhook.\n\n* **Discord**. Add a Discord webhook.\n\n## Alert conditions\n\n### Failed tests\n\nIf you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.\n\n<Tip>\n  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if they fail.\n</Tip>\n\n

In [25]:
def read_github_repository(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()

    documents = []
    with ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md', '.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc = {
                    'content': post.content,
                    'title': post.metadata.get('title'),
                    'description': post.metadata.get('description'),
                    'filename': file_path.split('/', 1)[-1]
                }
                documents.append(doc)

    return documents

In [26]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

Downloaded 95 documents


# Using github library instead

In [28]:
!uv add gitsource

Resolved 114 packages in 239ms
Prepared 1 package in 75ms
Installed 1 package in 31ms
 + gitsource==0.0.5


In [192]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [ ]:
# Library returns raw files 

In [62]:
document = files[10]

print(document.filename)
print(document.content[:250])

docs/library/output_formats.mdx
---
title: 'Output formats'
description: 'How to export the evaluation results.'
---

You can view or export Reports in multiple formats.

**Pre-requisites**:

* You know how to [generate Reports](/docs/library/report).

## Log to Workspace

You can 


In [63]:
post = frontmatter.loads(document.content)
data = post.to_dict()
data['filename'] = document.filename

In [ ]:
data

In [ ]:
# we can also use parse to get the same 

In [195]:
documents = [f.parse()for f in files]

In [ ]:
documents

# Indexing data with min search

In [71]:
!uv add minsearch

Resolved 124 packages in 1.30s
Prepared 10 packages in 24.68s
Installed 10 packages in 1.77s
 + cloudpickle==3.1.2
 + joblib==1.6.0
 + minsearch==0.2.0
 + narwhals==2.26.0
 + numpy==2.5.3
 + pandas==3.0.6
 + scikit-learn==1.9.1
 + scipy==1.18.1
 + stemlite==0.1.0
 + threadpoolctl==3.7.0


In [193]:
from minsearch import Index

In [79]:
help(Index)

Help on class Index in module minsearch.minsearch:

class Index(builtins.object)
 |  Index(text_fields, keyword_fields=None, numeric_fields=None, date_fields=None, vectorizer_params=None)
 |
 |  A simple search index using TF-IDF and cosine similarity for text fields,
 |  exact matching for keyword fields, and range filters for numeric and date fields.
 |
 |  Attributes:
 |      text_fields (list): List of text field names to index.
 |      keyword_fields (list): List of keyword field names to index.
 |      numeric_fields (list): List of numeric field names to index.
 |      date_fields (list): List of date field names to index.
 |      vectorizers (dict): Dictionary of TfidfVectorizer instances for each text field.
 |      keyword_df (pd.DataFrame): DataFrame containing keyword field data.
 |      numeric_df (pd.DataFrame): DataFrame containing numeric field data.
 |      date_df (pd.DataFrame): DataFrame containing date field data.
 |      text_matrices (dict): Dictionary of TF-IDF 

In [194]:
index = Index(
     text_fields=["title", "description", "content"],
     keyword_fields=["filename"]
)



In [83]:
index.fit(documents)

In [173]:
query = "LLM as Judge"
results = index.search(query, num_results=5)

In [174]:
type(results)

list

In [ ]:
results

# Chunking documents for better retrieval

In [109]:
len(files)

95

In [105]:
#type(files) - list
#type(files[0]) - dict
#files[0].   - content, file name, parse


list

In [131]:
#Analyze document sizes
doc_sizes = [(doc.filename, len(doc.content))  for doc in files]


In [133]:
doc_sizes[:5]

[('api-reference/endpoint/create.mdx', 53),
 ('api-reference/endpoint/delete.mdx', 60),
 ('api-reference/endpoint/get.mdx', 50),
 ('api-reference/introduction.mdx', 874),
 ('changelog/changelog.mdx', 5954)]

In [158]:
doc_sizes.sort(key=lambda x: x[1], reverse=True)

In [159]:
for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

metrics/all_metrics.mdx: 55085 characters
metrics/all_descriptors.mdx: 31976 characters
docs/platform/dashboard_panel_types.mdx: 31647 characters
docs/library/leftover_content.mdx: 28742 characters
metrics/customize_llm_judge.mdx: 26847 characters


In [167]:
documents = list(range(0,100))
window_size = 10
start =0
step =5

chunks = []

while start < len(documents):
    end = start + window_size
    chunk = documents[start:end]
    if len(chunk) < window_size:
        break
    chunks.append(chunk)
    print(chunk)
    start = start + step
        

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
[20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
[25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
[30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
[40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
[45, 46, 47, 48, 49, 50, 51, 52, 53, 54]
[50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
[55, 56, 57, 58, 59, 60, 61, 62, 63, 64]
[60, 61, 62, 63, 64, 65, 66, 67, 68, 69]
[65, 66, 67, 68, 69, 70, 71, 72, 73, 74]
[70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
[75, 76, 77, 78, 79, 80, 81, 82, 83, 84]
[80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
[85, 86, 87, 88, 89, 90, 91, 92, 93, 94]
[90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [168]:
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})

        start = end - step

        if end >= text_length:
            break

    return chunks

In [175]:
len(sliding_window(results[0]['content'], size=3000, step=2500))

39

In [213]:
#Create and indexable chunk
document_chunks = []
for doc in documents:
    if not doc.get('content'):
        continue        
    copy = doc.copy()
    content = copy.pop('content')
    chunks = (sliding_window(content, size=3000, step=2500)) 

    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk) 
    

In [216]:
document_chunks[10]

{'start': 1000,
 'content': 'on\neval_data = Dataset.from_pandas(\n    source_df,\n    data_definition=DataDefinition()\n)\n```\n\nTo map columns automatically, pass an empty `DataDefinition()` . Evidently will map columns:\n\n- By type (numerical, categorical).\n- By matching column names to roles (e.g., a column "target" treated as target).\n\nAutomation works in many cases, but manual mapping is more accurate. It is also necessary for evaluating prediction quality or handling text columns.\n\n<Note>\n  **How to set the data definition manually?** See the section below for available options.\n</Note>\n\n**Step 4. Run evals.** Once the **Dataset** object is ready, you can [add Descriptors ](/docs/library/descriptors) and[ run Reports](/docs/library/report).\n\n### Special cases\n\n**Working directly with pandas.DataFrame**. You can sometimes pass a `pandas.DataFrame` directly to `report.run()` without creating the Dataset object. This works for checks like numerical/categorical data s

In [ ]:
#Re-indexing the Chunks

In [217]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [222]:
query

'LLM as Judge'

In [220]:
results = chunk_index.search(query)

In [221]:
results

[{'start': 1000,
  'content': 'n.\n\n<Info>\n**Prefer videos?** We also have an extended code tutorial where we iteratively improve the prompt for LLM judge with a video walkthrough:  https://www.youtube.com/watch?v=kP_aaFnXLmY\n</Info>\n\n## Tutorial scope\n\nHere\'s what we\'ll do:\n\n- **Create an evaluation dataset**. Create a toy Q&A dataset.\n- **Create and run an LLM as a judge**. Design an LLM evaluator prompt.\n- **Evaluate the judge**. Compare the LLM judge\'s evaluations with manual labels.\n\nWe\'ll start with the reference-based evaluator that determines whether a new response is correct (it\'s more complex since it requires passing two columns to the prompt). Then, we\'ll create a simpler judge focused on verbosity.\n\nTo complete the tutorial, you will need:\n\n- Basic Python knowledge.\n- An OpenAI API key to use for the LLM evaluator.\n\nWe recommend running this tutorial in Jupyter Notebook or Google Colab to render rich HTML objects with summary results directly in a

#End to End 

In [224]:
from openai import OpenAI
import dotenv

In [226]:
dotenv.load_dotenv()
openai_client = OpenAI()

In [229]:
query = 'how do I implement llm as a judge?'

In [230]:
search_result = chunk_index.search(query, num_results=5)

In [231]:
import json

search_result_json = json.dumps(search_result, indent=2)

In [234]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [ ]:
# LLM function

In [240]:
def llm(user_prompt,instructions=None,model='gpt-4o-mini'):
    messages =[]

    if instructions is not None:
        messages.append({"role":"system","content":instructions})
        
    messages.append({"role":"user","content":user_prompt})
    


    response = openai_client.responses.create(
        model=model,
        input=messages
        
    )
    return response.output_text
    

In [241]:
answer = llm(user_prompt, instructions)

In [242]:
print(answer)

To implement a Large Language Model (LLM) as a judge, follow these steps:

1. **Set Up Your Environment**:
   - Ensure you have a basic understanding of Python.
   - Install the required Python library using:
     ```bash
     pip install evidently
     ```
   - Import the necessary modules in your Python script:
     ```python
     import pandas as pd
     import numpy as np
     from evidently import Dataset, DataDefinition, Report, BinaryClassification
     from evidently.descriptors import *
     from evidently.presets import TextEvals, ValueStats, ClassificationPreset
     from evidently.metrics import *
     from evidently.llm.templates import BinaryClassificationPromptTemplate
     ```

2. **Configure Your OpenAI API Key**:
   - Pass your OpenAI API key as an environment variable:
     ```python
     import os
     os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
     ```

3. **Create an Evaluation Dataset**:
   - Develop a toy Q&A dataset that includes:
     - **Questions**: Inputs to

In [246]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [247]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [248]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt

In [249]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [250]:
rag('how do I implement llm as a judge?')

'To implement an LLM (Large Language Model) as a judge, you can follow these steps:\n\n1. **Set Up Environment**:\n   - Ensure you have basic Python knowledge.\n   - Install the necessary library with:\n     ```bash\n     pip install evidently\n     ```\n   - Import required modules:\n     ```python\n     import pandas as pd\n     import numpy as np\n     from evidently import Dataset, DataDefinition, Report, BinaryClassification\n     from evidently.llm.templates import BinaryClassificationPromptTemplate\n     ```\n   - Set your OpenAI API key:\n     ```python\n     import os\n     os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n     ```\n   \n2. **Create an Evaluation Dataset**:\n   - Create a toy Q&A dataset that includes:\n     - **Questions**: Inputs sent to the LLM app.\n     - **Target Responses**: Approved responses considered accurate.\n     - **New Responses**: Imitated responses generated by the system.\n     - **Manual Labels**: Labels indicating if the response is correct or no

# Structured Output

In [253]:
!uv add pydantic

Resolved 124 packages in 1.63s
Audited 119 packages in 509ms


In [256]:
from pydantic import BaseModel
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


In [257]:
CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [258]:
#Structured Extraction

messages = [
    {"role": "system", "content": "Extract the event information."},
    {
        "role": "user",
        "content": "Alice and Bob are going to a science fair on Friday.",
    },
]

response = openai_client.responses.parse(
    model="gpt-4o-mini",
    input=messages,
    text_format=CalendarEvent,
)

In [261]:
response.output[0].content[0].parsed

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [262]:
response.output[0].content[0].text

'{"name":"Science Fair","date":"Friday","participants":["Alice","Bob"]}'

In [263]:
#short cut
event = response.output_parsed
event

CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])

In [ ]:
#unstructured output

In [264]:
response = openai_client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to a science fair on Friday.",
        },
    ],
)
print(response.output_text)

Event: Science Fair  
Participants: Alice and Bob  
Date: Friday  
